# 01 — Load Image (Sanity Check)

**Purpose:** Load a raw CMOS `.sif` image and verify it has the expected dimensions and metadata.  
No spectrum extraction — just confirm the file is readable and the image looks reasonable.

**When to run:** Any time you load a new image file for the first time, or after a hardware change.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from echelle_spectra.tools.echelle import read_image

%matplotlib inline

## Configuration

Set the path to your `.sif` image file.

In [ ]:
# Path to a raw CMOS image (.sif)
IMAGE_PATH = "/path/to/your/image.sif"

# Or use the bundled test image (CCD, 1024x1024):
# from pathlib import Path
# import echelle_spectra
# IMAGE_PATH = echelle_spectra._config['base_path'] / 'resources/test_data/CCD_Example.SIF'

## Load Image

In [ ]:
images, info = read_image(IMAGE_PATH, spec="black")

print("Image shape (frames, rows, cols):", images.shape)
print("Frames:    ", images.shape[0])
print("Rows:      ", images.shape[1], "  (order direction)")
print("Columns:   ", images.shape[2], "  (wavelength direction)")
print()
print("Metadata:")
for k, v in info.items():
    print(f"  {k}: {v}")

## Expected CMOS Dimensions

| Detector | Rows | Cols | Orders |
|----------|------|------|--------|
| CCD  | 1024 | 1024 | 28     |
| CMOS | 2160 | 2560 | 29     |

If the shape does not match, check binning settings or whether the image was cropped.

In [ ]:
nrows, ncols = images.shape[1], images.shape[2]

if nrows == 2160 and ncols == 2560:
    print("✓ CMOS image: 2560 × 2160, 29 orders")
elif nrows == 1024 and ncols == 1024:
    print("✓ CCD image: 1024 × 1024, 28 orders")
else:
    print(f"⚠ Unexpected dimensions: {ncols} × {nrows}")
    print("  Check binning or crop settings.")

## Display Raw Image

In [ ]:
frame = 0  # display the first frame
img = images[frame]

fig, ax = plt.subplots(figsize=(14, 6))
norm = mcolors.LogNorm(vmin=max(img.min(), 1), vmax=img.max())
im = ax.imshow(img, origin="lower", cmap="inferno", norm=norm, aspect="auto")
plt.colorbar(im, ax=ax, label="Counts")
ax.set_title(f"Raw CMOS image — frame {frame}")
ax.set_xlabel("Column (wavelength direction)")
ax.set_ylabel("Row (order direction)")
plt.tight_layout()
plt.show()

## Intensity Statistics

In [ ]:
print(f"Min:    {img.min():.1f}")
print(f"Max:    {img.max():.1f}")
print(f"Mean:   {img.mean():.1f}")
print(f"Median: {np.median(img):.1f}")

if img.max() > 60000:
    print("⚠ Possible saturation (>60000 counts). Check exposure time.")
else:
    print("✓ No saturation detected.")